## Unzip the data
Extract the contents of the zip file.


In [ ]:
import pandas as pd
from pathlib import Path

folder = Path("../ALLDataGross/healthyCohort")
dataframes = []

for file in sorted(folder.glob("*.dpt")):
    try:
        # whitespace flexible (tabs or spaces)
        df = pd.read_csv(file, sep=r"\s+", engine="python")
        if df.shape[1] != 2:
            df = pd.read_csv(file, sep=r",", engine="python")
        dataframes.append((file.stem, df))
        print(f"✅ Loaded {file.name} with {df.shape[1]} columns")
        df.columns = ["Wavenumber", "Intensity"]
    except Exception as e:
        print(f"❌ Error loading {file.name}: {e}")


# Example: access in sequence
for name, df in dataframes:
    print(f"\n{name}:")
    display(df.head())


## Load the data

### Subtask:
Load the data from the unzipped file(s) into a pandas DataFrame.


**Reasoning**:
List the files in the extraction directory to identify the data file(s) and then load the data from the identified file(s) into a pandas DataFrame.



## Analyze the data

### Subtask:
Perform some basic analysis on the data, such as calculating summary statistics or creating visualizations.


# Task
Analyze and visualize each dataframe in the `dataframes` dictionary.

In [ ]:
import matplotlib.pyplot as plt
import math
import numpy as np

# Create a single figure and axis for all plots
fig, ax = plt.subplots(figsize=(12, 8))

for filename, df in dataframes:
    ax.plot(df['Wavenumber'], df['Intensity'], label=filename) # Plot each DataFrame with a label

ax.set_title("All DataFrames")
ax.set_xlabel("Wavenumber(cm-1)")
ax.set_ylabel("Intensity")
ax.grid(True)

# Add a legend to identify the curves
ax.legend()

# plt.xlim(3750,3950)
# plt.ylim(-1,1)

plt.tight_layout() # Adjust layout
plt.show()

Base Line Correction
Overall pipeline

1. Inspect raw data: plot intensity vs wavenumber; check SNR, water/CO₂ features, fringes.

2. Pre-clean (optional): remove obvious cosmic spikes, spike interpolation, smoothing (Savitzky-Golay with small window).

3. Atmospheric removal / region masking: either subtract measured background (if you have it) or exclude/handle strong water/CO₂ bands (or use targeted water-correction methods). For breath IR you may need selective region processing.
ScienceDirect

4. Baseline correction: try multiple methods (AsLS, airPLS, SNIP, rubberband). Keep parameters in a small grid so you can compare.

5. Normalization: e.g., Standard Normal Variate (SNV), area normalization, or peak normalization depending on downstream goal.

6. QA & visualization: overlay raw, baseline, corrected; display residuals and preserved peak integrals.

7. Quantitative checks / choose best

## Step 1: Inspect Data in Detail.
X-axis = wavenumber (cm⁻¹).

Y-axis = absorbance (higher = stronger absorption).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def inspect_ir_spectrum(wavenumber, intensity,
                        flat_region=(2200, 2000),  # region with no strong peaks
                        peak_region=(2350, 2400),  # CO₂ band region for signal estimate
                        plot_title="IR Breath Spectrum Inspection"):
    """
    Inspect raw IR spectrum:
      - Plot intensity vs wavenumber
      - Highlight water & CO₂ regions
      - Estimate SNR using a flat region for noise and CO₂ peak for signal

    Parameters
    ----------
    wavenumber : array-like
        Wavenumber axis (cm⁻¹).
    intensity : array-like
        Absorbance (or transmittance) values.
    flat_region : tuple (min, max)
        Wavenumber range to estimate noise (default: 2200–2000 cm⁻¹).
    peak_region : tuple (min, max)
        Wavenumber range to estimate signal peak (default: 2350–2400 cm⁻¹).
    plot_title : str
        Title for the plot.

    Returns
    -------
    snr : float
        Estimated signal-to-noise ratio.
    """

    wn = np.array(wavenumber)
    y = np.array(intensity)

    # Ensure wavenumber is increasing (some instruments output decreasing)
    if wn[0] > wn[-1]:
        wn = wn[::-1]
        y = y[::-1]

    # --- Noise estimation ---
    mask_noise = (wn >= flat_region[1]) & (wn <= flat_region[0])
    noise_std = np.std(y[mask_noise]) if np.any(mask_noise) else np.nan

    # --- Signal estimation (use CO₂ band or another strong band) ---
    mask_peak = (wn >= peak_region[0]) & (wn <= peak_region[1])
    peak_height = np.max(y[mask_peak]) - np.min(y[mask_noise]) if np.any(mask_peak) else np.nan

    snr = peak_height / noise_std if noise_std > 0 else np.nan

    # --- Plot ---
    plt.figure(figsize=(10, 5))
    plt.plot(wn, y, label="Raw spectrum", color="black")

    # Highlight water vapor regions
    plt.axvspan(3700, 3100, color="blue", alpha=0.3, label="H₂O (O-H stretch)")
    plt.axvspan(1650-50, 1650+50, color="skyblue", alpha=0.3)  # H₂O bending

    # Highlight CO₂ region
    plt.axvspan(peak_region[0], peak_region[1], color="lightcoral", alpha=0.3, label="CO₂ band")

    # Flat region used for noise
    plt.axvspan(flat_region[1], flat_region[0], color="yellow", alpha=0.3, label="Noise region")

    plt.gca().invert_xaxis()  # IR convention: decreasing axis
    plt.xlabel("Wavenumber (cm⁻¹)")
    plt.xlim(3100,3000)
    plt.ylabel("Absorbance")
    plt.title(plot_title)
    plt.legend()
    plt.show()

    print(f"Estimated SNR (peak {peak_region} / noise {flat_region}): {snr:.2f}")

    return snr


In [ ]:
filename, df = dataframes[1]
snr_value = inspect_ir_spectrum(df['Wavenumber'], df['Intensity'])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def _robust_std(x):
    # MAD-based robust std estimate (consistent with normal dist.)
    mad = np.median(np.abs(x - np.median(x)))
    return 1.4826 * mad if mad > 0 else np.std(x)

def estimate_snr(
    wn, y,
    noise_region=(2200, 2000),
    signal_region=(2350, 2400),
    detrend_noise=True,
    use_robust_noise=True
):
    wn = np.asarray(wn)
    y  = np.asarray(y)
    if wn[0] > wn[-1]:
        wn = wn[::-1]; y = y[::-1]

    # masks (assume region given as (high, low))
    n_lo, n_hi = noise_region[1], noise_region[0]
    s_lo, s_hi = signal_region[0], signal_region[1]
    noise_mask = (wn >= n_lo) & (wn <= n_hi)
    signal_mask = (wn >= s_lo) & (wn <= s_hi)

    if not np.any(noise_mask) or not np.any(signal_mask):
        return np.nan, np.nan, np.nan

    noise_x = wn[noise_mask]
    noise_y = y[noise_mask]

    # optional linear detrend in the noise window
    if detrend_noise and len(noise_x) >= 3:
        A = np.vstack([noise_x, np.ones_like(noise_x)]).T
        m, b = np.linalg.lstsq(A, noise_y, rcond=None)[0]
        noise_y = noise_y - (m * noise_x + b)

    noise_std = _robust_std(noise_y) if use_robust_noise else np.std(noise_y)

    sig_y = y[signal_mask]
    signal_height = np.max(sig_y) - np.min(noise_y) if len(sig_y) else np.nan

    snr = signal_height / noise_std if noise_std > 0 else np.nan
    return snr, signal_height, noise_std

def batch_snr_report(
    spectra_dict,
    noise_region=(2200, 2000),
    signal_region=(2350, 2400),
    detrend_noise=True,
    use_robust_noise=True,
    plot_hist=True
):
    """
    spectra_dict: {sample_id: (wn_array, y_array)}
    """
    rows = []
    for sid, (wn, y) in spectra_dict.items():
        snr, sig, nstd = estimate_snr(
            wn, y,
            noise_region=noise_region,
            signal_region=signal_region,
            detrend_noise=detrend_noise,
            use_robust_noise=use_robust_noise
        )
        rows.append((sid, snr, sig, nstd))

    # Print a simple table
    print("ID\tSNR\tSignalHeight\tNoiseStd")
    for r in rows:
        print(f"{r[0]}\t{r[1]:.2f}\t{r[2]:.4g}\t{r[3]:.4g}")

    # Flag potential issues
    bad = [r for r in rows if (np.isnan(r[1]) or r[1] < 20)]
    if bad:
        print("\n⚠️ Low/invalid SNR samples:")
        for r in bad:
            print(f"  - {r[0]} (SNR={r[1]})")

    if plot_hist:
        snrs = np.array([r[1] for r in rows if np.isfinite(r[1])])
        if snrs.size:
            plt.figure(figsize=(6,4))
            plt.hist(snrs, bins=20)
            plt.xlabel("SNR")
            plt.ylabel("Count")
            plt.title("SNR distribution (robust, detrended noise)")
            plt.show()

    return rows


In [ ]:
# Reformat dataframes to match the expected input format for batch_snr_report
spectra_data = {filename: (df['Wavenumber'].values, df['Intensity'].values) for filename, df in dataframes}

# Call the batch_snr_report function
snr_results = batch_snr_report(spectra_data)

##Step 2:

In [ ]:
import numpy as np

def hampel_flags(y, window=7, n_sigma=6.0):
    """
    Return a boolean mask of spike locations using a Hampel filter.
    window: odd number of points in the local window (typ 7–15).
    n_sigma: threshold in robust std units (typ 5–8 for spectra).
    """
    y = np.asarray(y, float)
    n = len(y)
    k = window // 2
    flags = np.zeros(n, dtype=bool)
    # 1.4826 * MAD ≈ robust std for normal data
    for i in range(n):
        lo = max(0, i - k)
        hi = min(n, i + k + 1)
        med = np.median(y[lo:hi])
        mad = np.median(np.abs(y[lo:hi] - med))
        if mad == 0:
            continue
        robust_std = 1.4826 * mad
        if np.abs(y[i] - med) > n_sigma * robust_std:
            flags[i] = True
    return flags

def interpolate_spikes(x, y, flags):
    """
    Linearly interpolate across flagged points.
    """
    x = np.asarray(x, float)
    y = np.asarray(y, float).copy()
    if flags.any():
        good = ~flags
        y[flags] = np.interp(x[flags], x[good], y[good])
    return y


In [ ]:
# Assuming you want to process the first dataframe in the dictionary
filename, df = dataframes[0]
wavenumber = df['Wavenumber'].values
intensity = df['Intensity'].values

# Identify spikes using Hampel filter
spike_flags = hampel_flags(intensity)

# Print indices where spikes were flagged
spike_flag_indices = [i for i, flag in enumerate(spike_flags) if flag]
print(f"Indices of flagged spikes in {filename}: {spike_flag_indices}")

# Interpolate across flagged spikes
corrected_intensity = interpolate_spikes(wavenumber, intensity, spike_flags)

# You can now use corrected_intensity for further analysis or plotting
# For example, plot the original and corrected data:
plt.figure(figsize=(10, 5))
plt.plot(wavenumber, intensity, label="Original", alpha=0.7)
plt.plot(wavenumber, corrected_intensity, label="Spike Corrected")
plt.scatter(wavenumber[spike_flags], intensity[spike_flags], color='red', label='Spikes', zorder=5) # Highlight spikes
plt.gca().invert_xaxis()
plt.xlabel("Wavenumber (cm⁻¹)")
plt.ylabel("Absorbance")
plt.title(f"Spike Correction for {filename}")
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

for idx in [638, 644, 651, 657, 663, 676, 751, 758, 764, 771, 778, 784, 785, 791, 876, 1991, 1996, 2409, 2419, 7422, 7441, 7446, 7454, 7455, 7464, 7465, 12982, 13289, 13296, 13393, 13399, 13404, 13409, 13419, 13424, 13461]:  # pick a few flagged ones
    sl = slice(max(idx-10,0), idx+11)
    plt.figure()
    plt.plot(wavenumber[sl], intensity[sl], 'o-', label="Raw")
    plt.axvline(wavenumber[idx], color='r', linestyle='--', label='Flagged')
    plt.legend()
    plt.title(f"Check spike at index {idx}, wn={wavenumber[idx]:.1f}")
    plt.show()


#Step 3: Region masking (for C0$_2$ or H$_2$0 bands)

#Step 4: Baseline Correction
**1. AsLS (Asymmetric Least Squares)**
- Fits a smooth baseline curve.
- Good default choice.

**2. airPLS (adaptive iteratively reweighted PLS)**
- Iteratively adjusts weights so peaks don’t bias the baseline.

**3. SNIP (Statistics-sensitive Non-linear Iterative Peak-clipping)**
- Works well for sharp spectral peaks (less common in IR).

**4. Rubberband method**
- “Stretch a rubber band” under the spectrum.
- Simple, intuitive, but less flexible.

In [ ]:
# --- Step 4: Baseline Correction Toolkit ---

import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.sparse.linalg import spsolve

# --------------------------
# 1. Asymmetric Least Squares (AsLS)
# --------------------------
def baseline_asls(y, lam=1e6, p=0.01, niter=10):
    L = len(y)
    D = sparse.diags([1,-2,1],[0,-1,-2], shape=(L, L-2))
    w = np.ones(L)
    for i in range(niter):
        W = sparse.spdiags(w, 0, L, L)
        Z = W + lam * D.dot(D.transpose())
        z = spsolve(Z, w*y)
        w = p * (y > z) + (1-p) * (y < z)
    return z

# --------------------------
# 2. airPLS
# --------------------------
def baseline_airpls(y, lam=1e5, porder=1, itermax=50):
    m = len(y)
    w = np.ones(m)
    for i in range(1, itermax+1):
        z = baseline_asls(y, lam=lam, p=0.01, niter=10)
        d = y - z
        dn = d[d < 0]
        m_dn = np.mean(dn) if len(dn) > 0 else 0
        s_dn = np.std(dn) if len(dn) > 0 else 0
        w[d >= 0] = 0
        w[d < 0] = np.exp(i * (d[d < 0] - (2*s_dn - m_dn)) / (s_dn+1e-8))
        w = np.clip(w, 0, 1)
    return z

# --------------------------
# 3. SNIP (simplified version)
# --------------------------
def baseline_snip(y, iterations=20):
    """
    Simplified SNIP baseline estimator.
    """
    y_snip = np.copy(y)
    L = len(y)
    for k in range(iterations):
        for i in range(k+1, L-k-1):
            y_snip[i] = min(y_snip[i], (y_snip[i-k-1] + y_snip[i+k+1]) / 2)
    return y_snip

# --------------------------
# 4. Rubberband method
# --------------------------
from scipy.spatial import ConvexHull

def baseline_rubberband(x, y):
    pts = np.array([x, y]).T
    hull = ConvexHull(pts)
    mask = hull.vertices
    mask = mask[np.argsort(x[mask])]
    baseline = np.interp(x, x[mask], y[mask])
    return baseline

# --------------------------
# Demo on one spectrum
# --------------------------

# Example: load a spectrum (replace with your real data)
x = wavenumber  # wavenumber
y = intensity # fake spectrum

# Compute baselines
# baseline1 = baseline_asls(y, lam=1e6, p=0.01)
# baseline2 = baseline_airpls(y, lam=1e5)
baseline3 = baseline_snip(y, iterations=30)
# baseline4 = baseline_rubberband(x, y)

# Apply correction
methods = {
    # "AsLS": y - baseline1,
    # "airPLS": y - baseline2,
    "SNIP": y - baseline3,
    # "Rubberband": y - baseline4,
}

# Plot comparison
plt.figure(figsize=(12,6))
# plt.plot(x, y, 'k-', alpha=0.6, label="Raw")

for name, y_corr in methods.items():
    plt.plot(x, y_corr, label=name)

# plt.xlim(3750,3950)
plt.ylim(-0.01,0.01)

plt.xlabel("Wavenumber (cm$^{-1}$)")
plt.ylabel("Intensity (a.u.)")
plt.legend()
plt.title("Baseline Correction Methods Comparison")
plt.show()


In [ ]:
# --- Evaluation helpers ---

def robust_std(x):
    x = np.asarray(x, float)
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    return 1.4826 * mad if mad > 0 else x.std()

def region_mask(x, region):
    lo, hi = min(region), max(region)
    return (x >= lo) & (x <= hi)

def baseline_flatness(x, y_corr, flat_regions):
    """
    Compute robust std in one or multiple flat regions and average them.
    flat_regions: list of (lo, hi)
    """
    vals = []
    for reg in flat_regions:
        m = region_mask(x, reg)
        if m.any():
            vals.append(robust_std(y_corr[m]))
    return np.mean(vals) if vals else np.nan

def peak_metrics(x, y_before, y_after, peak_regions):
    """
    Compute peak height & area before/after, and percent bias for each region.
    Returns list of dicts.
    """
    out = []
    for reg in peak_regions:
        m = region_mask(x, reg)
        if not m.any():
            out.append({"region": reg, "height_bias_pct": np.nan, "area_bias_pct": np.nan})
            continue
        h0 = np.max(y_before[m]) - np.min(y_before[m])
        h1 = np.max(y_after[m])  - np.min(y_after[m])
        a0 = np.trapz(y_before[m], x[m])
        a1 = np.trapz(y_after[m],  x[m])
        hb = 100.0 * (h1 - h0) / h0 if h0 != 0 else np.nan
        ab = 100.0 * (a1 - a0) / a0 if a0 != 0 else np.nan
        out.append({"region": reg, "height_bias_pct": hb, "area_bias_pct": ab})
    return out

def noise_change(x, y_before, y_after, flat_regions):
    n0 = baseline_flatness(x, y_before, flat_regions)
    n1 = baseline_flatness(x, y_after,  flat_regions)
    ratio = (n1 / n0) if (n0 is not None and n0 not in [0, np.nan]) else np.nan
    return n0, n1, ratio


In [ ]:
import matplotlib.pyplot as plt

# Example regions (tune to your data)
FLAT_REGIONS  = [(2200, 2000), (2500, 2450)]        # peak-free areas for noise/flatness
PEAK_REGIONS  = [(2350, 2400)]                      # CO₂ reference; add biomarker peaks if known

def evaluate_methods(x, y_raw, vapor_pressure_mbar, methods_dict,
                     apply_vp_norm=False,
                     flat_regions=FLAT_REGIONS,
                     peak_regions=PEAK_REGIONS):
    """
    methods_dict: {"AsLS": baseline_func, ...}
    baseline_func should return 'baseline' (same length), we'll do y_corr = y_raw - baseline
    """
    # keep a copy of raw for "before" metrics
    report = []
    curves  = {}

    for name, bl_fun in methods_dict.items():
        bl = bl_fun(x, y_raw)                  # baseline estimate
        y_corr = y_raw - bl

        # optional: apply vapor-pressure 500 mbar normalization right after correction
        # y_eval = normalize_vapor_pressure(y_corr, vapor_pressure_mbar, 500.0) if apply_vp_norm else y_corr
        y_eval = y_corr


        # metrics
        flat = baseline_flatness(x, y_eval, flat_regions)
        peaks = peak_metrics(x, y_raw, y_eval, peak_regions)
        n0, n1, nratio = noise_change(x, y_raw, y_eval, flat_regions)

        report.append({
            "method": name,
            "flat_std_after": flat,
            "noise_std_before": n0,
            "noise_std_after": n1,
            "noise_ratio_after_over_before": nratio,
            "peak_metrics": peaks
        })
        curves[name] = (bl, y_eval)

    return report, curves

def pretty_print_report(report):
    for r in report:
        print(f"\n=== {r['method']} ===")
        print(f"Flatness (robust std) after: {r['flat_std_after']:.4g}")
        print(f"Noise std before: {r['noise_std_before']:.4g} -> after: {r['noise_std_after']:.4g} (ratio {r['noise_ratio_after_over_before']:.3g})")
        for pm in r["peak_metrics"]:
            lo, hi = pm["region"]
            print(f"Peak {lo}-{hi} cm^-1: height bias {pm['height_bias_pct']:.2f}%, area bias {pm['area_bias_pct']:.2f}%")


In [ ]:
# You should already have baseline functions defined, e.g.:
# baseline_asls, baseline_airpls, baseline_snip, baseline_rubberband

# def bl_asls(x, y):       return baseline_asls(y, lam=1e6, p=0.01)
# def bl_airpls(x, y):     return baseline_airpls(y, lam=1e5)
def bl_snip(x, y):       return baseline_snip(y, iterations=30)
# def bl_rubber(x, y):     return baseline_rubberband(x, y)

methods = {
    # "AsLS": bl_asls,
    # "airPLS": bl_airpls,
    "SNIP": bl_snip,
    # "Rubberband": bl_rubber
}

# Load data for the first spectrum in dataframes
filename, df = dataframes[3] # Choose which dataframe to use
x = df['Wavenumber'].values
y_raw = df['Intensity'].values

# Define vapor pressure (replace with your actual value)
vapor_pressure_mbar = 500 # Example value in mbar

report, curves = evaluate_methods(x, y_raw, vapor_pressure_mbar, methods)
pretty_print_report(report)

# Optional: quick overlay plot

plt.figure(figsize=(12,8))
plt.plot(x, y_raw, label="Raw", alpha=0.5)
for name, (bl, y_eval) in curves.items():
    plt.plot(x, y_eval, label=f"{name} (corr+VP500)")
plt.gca().invert_xaxis()
# plt.xlim(2250,2000)
plt.xlabel("Wavenumber (cm$^{-1}$)"); plt.ylabel("Absorbance")
plt.title("Baseline-corrected & VP-normalized spectra")
plt.legend(); plt.show()